# Comparación entre los códigos que analizan la postura de un robot DOFBot y un SCARA

### **Documentación del código que controla el brazo de un DOFBot y su gripper. dofbot_sequence_py.py**

*Roberto Miranda Meza*

*Grupo 1*

![Robot DOFBot, tiene 5 juntas rotacionales a diferencia de un robot SCARA con 3](Imagenes/DOFBotImagen.jpg)

Un Robot DOFBot, tiene 5 juntas rotacionales a diferencia de un robot SCARA con 3 juntas rotacionales. Aquí el análisis del robot se divide en 2, una parte es desde la base que gira hasta el efector final del tercer eslabón y la otra son las aperturas del gripper. A continuación se muestra la documentación del código que resuelve la postura del robot DOFBot explicado por módulos:

In [ ]:
#!/usr/bin/env python3
#El Shebang indica al sistema que se requiere Python 3 para ejecutar el script.

import rclpy #Se importa la librería de clientes de ROS 2.
from rclpy.node import Node #Se importa la clase Node.
from trajectory_msgs.msg import JointTrajectory, JointTrajectoryPoint #Se importan las funciones para enviar mensajes de trayectoria.
from builtin_interfaces.msg import Duration #Se importa Duration para especificar tiempos dentro de los mensajes.

import time
from math import cos, sin, acos, asin, atan2, sqrt #Se importan funciones trigonométricas a utilizar en el cálculo de la posición.

In [ ]:
class DofbotControlNode(Node): #Se crea la clase DofbotControlNode
    def __init__(self): 
        super().__init__("dofbot_tray_control_node") #Se crea un nodo llamado 'dofbot_tray_control_node'
        self.lamda_ = 0 #se crea el parámetro lamda_ que recorre la trayectoria paso a paso
        topic_dofbot_ = "/dofbot_trajectory_controller/joint_trajectory" #se genera un tópico llamado topic_dofbot donde se guarda la trayectoria del brazo
        topic_gripper_ = "/dofbot_gripper_controller/joint_trajectory" #se genera otro tópico topic_gripper donde se guarda la trayectoria del gripper
        self.dofbot_publisher_ = self.create_publisher(JointTrajectory, topic_dofbot_, 10) #Se crea un objeto publicador que publica la trayectoria del brazo 10 veces por segundo
        self.dofbot_joints_ = ['arm_joint_01', 'arm_joint_02','arm_joint_03', 'arm_joint_04', 'arm_joint_05'] #se enlistan las juntas que componen al robot DOFBot
        
        self.gripper_publisher_ = self.create_publisher(JointTrajectory, topic_gripper_, 10) #se crea un objeto publicador de la trayectoria del gripper (efector final) que publica en el tópico 'topic_gripper_' 10 veces por segundo
        self.gripper_joints_ = ['grip_joint', 'rfinger_joint_01','rfinger_joint_02', 'lfinger_grip_joint_01','lfinger_grip_joint_02', 'lfinger_grip_joint_03'] #se enlistan las juntas del gripper del DOFBot
        
        
        self.timer_ = self.create_timer(0.5, self.timer_callback) #es un temporizador que llama a self.timer_callback() cada 0.5 s
        self.get_logger().info('Nodo de control del dofbot en funcionamiento') #envía un mensaje en la terminal que avisa cuando el nodo del DOFBot está en funcionamiento


In [ ]:
    def timer_callback(self): #se define la función timer_callback()
        dofbot_msg = JointTrajectory() #la variable dofbot_msg es un objeto de la clase JointTrajectory()
        dofbot_msg.joint_names = self.dofbot_joints_ #se guardan los nombres de las juntas en la lista joint_names del objeto dofbot_msg
        dofbot_point = JointTrajectoryPoint() #se genera un punto de trayectoria

        gripper_msg = JointTrajectory() #la variable gripper_msg es un objeto de la clase JointTrajectory()
        gripper_msg.joint_names = self.gripper_joints_ #se guardan los nombres de las juntas del gripper del DOFBot en la lista joint_names del objeto gripper_msg
        gripper_point = JointTrajectoryPoint() #se genera un punto de trayectoria

        #Lo siguiente es una secuencia de acciones que el robot lleva a cabo conforme el valor de lamda aumenta
        if self.lamda_ == 0:
            # El griper se abre o está abierto
            gstate = 1.57 #ángulo de apertura en radianes, en grados son aproximadamente 90°
            gripper_st = gripper_state(gstate) #de la función gripper_state se le asigna como argumento una lista de las posiciones de las articulaciones del gripper, en este caso 90°
            gripper_point.positions = gripper_st #el gripper toma las posiciones designadas
            gripper_point._time_from_start = Duration(sec=1) #indica que dura 1 segundo en ejecutarse la acción
            gripper_msg.points.append(gripper_point) #la última posición se guarda en gripper_msg donde se guardan las trayectorias o posturas del gripper
            self.gripper_publisher_.publish(gripper_msg) #se publica en el objeto gripper_publisher_ el valor de esta posición que publicará en el topic_gripper su posición
            self.get_logger().info('Gripper open') #se publica en la terminal un mensaje que avisa que el gripper está abierto
            self.get_logger().info('poture {}'.format(gripper_st)) #se publica en la terminal la postura del gripper
            time.sleep(5) #se pausa por 5 segundos para que suceda el movimiento
            self.lamda_ +=1 #aumenta el valor de self.lamda en uno para continuar el movimiento

In [ ]:

        elif self.lamda_ == 1:
            # El gripper se cierra
            gstate_2 = 0 #ahora el ángulo de apertura es de 0°
            gripper_st = gripper_state(gstate_2) #de la función gripper_state se asigna 0 radianes como argumento
            gripper_point.positions = gripper_st
            gripper_point._time_from_start = Duration(sec=1)
            gripper_msg.points.append(gripper_point)
            self.gripper_publisher_.publish(gripper_msg)
            self.get_logger().info('Gripper close')
            self.get_logger().info('poture {}'.format(gripper_st))
            time.sleep(5) #sucedió lo mismo que el caso anterior solo con mensajes nuevos y que el gripper se cerró
            self.lamda_ +=1 #el valor de self.lamda aumenta en uno
        
        elif self.lamda_ == 2:
            # El gripper se vuelve a abrir y se repite lo sucedido cuando lamda era igual a 0
            gstate = 1.57
            gripper_st = gripper_state(gstate)
            gripper_point.positions = gripper_st
            gripper_point._time_from_start = Duration(sec=1)
            gripper_msg.points.append(gripper_point)
            self.gripper_publisher_.publish(gripper_msg)
            self.get_logger().info('Gripper open')
            self.get_logger().info('poture {}'.format(gripper_st))
            time.sleep(5)
            self.lamda_ +=1 #aumenta el valor de self.lamda en uno


In [ ]:
        elif self.lamda_ == 3:
            #Ahora el DOFBot adopta una postura
            x_1 = 0.2
            y_1 = 0.0
            z_1 = 0.05 #estos son los valores de las coordenadas del eslabón 1 o brazo 1 en la primer junta
            theta_p_1 = 3.1416*(3/4) #ángulo de la junta 1
            theta_g_1 = 0.0 #ángulo de rotación del gripper
            solution_pos = dofbot_ink(x_1, y_1, z_1, theta_p_1, theta_g_1) #solution_pos se le devuelven los valores de los ángulos de todas las juntas, esto es calculado por la función dofbot_ink
            dofbot_point.positions = solution_pos #las posiciones de todos los eslabones se definen con los valores de las juntas
            dofbot_point.time_from_start = Duration(sec=2) #indica que el movimiento debe durar 2 segundos
            dofbot_msg.points.append(dofbot_point) #se le agrega el punto de trayectoria dofbot_point a dofbot_msg
            self.dofbot_publisher_.publish(dofbot_msg) #se publica el mensaje en el tópico topic_dofbot_
            self.get_logger().info('poture {}'.format(solution_pos)) #se publica un mensaje en la terminal que indica los valores de postura que deben alcanzar las juntas
            time.sleep(15) #espera 15 segundos para que se realice el movimiento
            self.lamda_ += 1 #self.lamda aumenta en 1

        elif self.lamda_ == 4:
            # El gripper se cierra, sucede lo mismo como cuando lamda tenía el valor de 1
            gstate_2 = 0
            gripper_st = gripper_state(gstate_2)
            gripper_point.positions = gripper_st
            gripper_point._time_from_start = Duration(sec=2)
            gripper_msg.points.append(gripper_point)
            self.gripper_publisher_.publish(gripper_msg)
            self.get_logger().info('Gripper close')
            self.get_logger().info('poture {}'.format(gripper_st)) #la posición del DOFBot debió de cambiar con respecto a cuando self.lamda_ valía 1
            time.sleep(10) #esta vez espera 10 segundos
            self.lamda_ +=1

In [ ]:
        elif self.lamda_ == 5:
            # El DOFBot adopta una segunda postura
            x_2 = 0.15
            y_2 = 0.0
            z_2 = 0.11 #estos son las coordenadas del efector final del eslabón 2
            theta_p_2 = 3.1416*(3/4) #el valor del ángulo de giro de la segunda junta
            theta_g_2 = 0 #el valor del ángulo de giro del gripper
            solution_pos = dofbot_ink(x_2, y_2, z_2, theta_p_2, theta_g_2) #se soluciona la posición
            dofbot_point.positions = solution_pos #los valores de posición se guardan en el punto de trayectoria
            dofbot_point.time_from_start = Duration(sec=2) #el movimiento dura 2 segundos
            dofbot_msg.points.append(dofbot_point) #se guarda la trayectoria en dofbot_msg
            self.dofbot_publisher_.publish(dofbot_msg) #se publica el mensaje en el tópico del dofbot, topic_dofbot_
            self.get_logger().info('poture {}'.format(solution_pos)) #se publica un mensaje en la terminal indicando la postura del robot
            time.sleep(15) #se dejan 15 segundos para realizar todo el movimiento
            self.lamda_ += 1 #self.lamda aumenta en 1

        elif self.lamda_ == 6:
            # El DOFBot adopta una tercer postura repitiendose los pasos de modo similar a otros valores de self.lamda_
            x_3 = 0.15
            y_3 = 0.15
            z_3 = 0.11
            theta_p_3 = 3.1416*(3/4) 
            theta_g_3 = 0 
            solution_pos = dofbot_ink(x_3, y_3, z_3, theta_p_3, theta_g_3)
            dofbot_point.positions = solution_pos
            dofbot_point.time_from_start = Duration(sec=2)
            dofbot_msg.points.append(dofbot_point)
            self.dofbot_publisher_.publish(dofbot_msg)
            self.get_logger().info('poture {}'.format(solution_pos))
            time.sleep(15)
            self.lamda_ += 1

In [ ]:
        elif self.lamda_ == 7:
            # El DOFBot adopta una cuarta postura repitiendose los pasos de modo similar a otros valores de self.lamda_
            # La diferencia con otros valores de lamda es que para 6 y 7 únicamente se mueve la junta 3
            x_3 = 0.15
            y_3 = 0.15
            z_3 = 0.05
            theta_p_3 = 3.1416*(3/4) 
            theta_g_3 = 0 
            solution_pos = dofbot_ink(x_3, y_3, z_3, theta_p_3, theta_g_3)
            dofbot_point.positions = solution_pos
            dofbot_point.time_from_start = Duration(sec=2)
            dofbot_msg.points.append(dofbot_point)
            self.dofbot_publisher_.publish(dofbot_msg)
            self.get_logger().info('poture {}'.format(solution_pos))
            time.sleep(15)
            self.lamda_ += 1

        elif self.lamda_ == 8:
            # El gripper se abre nuevamente 90°
            gstate = 1.57
            gripper_st = gripper_state(gstate)
            gripper_point.positions = gripper_st
            gripper_point._time_from_start = Duration(sec=2)
            gripper_msg.points.append(gripper_point)
            self.gripper_publisher_.publish(gripper_msg)
            self.get_logger().info('Gripper open')
            self.get_logger().info('poture {}'.format(gripper_st))
            time.sleep(10)
            self.lamda_ +=1

        elif self.lamda_ == 9:
            #en este último elif se decide llevar todas las posiciones del DOFBot a cero, x_P, y_P, z_P, theta_1_P y theta_g que compete al gripper
            solution_pos = [ float(0.0), float(0.0), float(0.0), float(0.0), float(0.0)]
            dofbot_point.positions = solution_pos
            dofbot_point.time_from_start = Duration(sec=2)
            dofbot_msg.points.append(dofbot_point)
            self.dofbot_publisher_.publish(dofbot_msg)
            self.get_logger().info('poture {}'.format(solution_pos))
            time.sleep(10)

In [ ]:

def dofbot_ink(x_P, y_P, z_P, theta_1_P, theta_g): #esta función resuelve la cinemática inversa del DOFBot
    # Parametros físicos del robot
    z_0_1 = 0.105 #la posición en Z de la primer junta
    L_1 = 0.084
    L_2 = 0.084
    L_3 = 0.115 #Son las longitudes de los 3 eslabones o brazos del DOFBot

    theta_1 = atan2(y_P, x_P) #este es el ángulo de giro 'base' del DOFBot
    aux_x = sqrt(pow(x_P, 2) + pow(y_P, 2)) - L_3*sin(theta_1_P)
    aux_z = z_P - z_0_1 -L_3*cos(theta_1_P)
    norm_4_P = sqrt(pow(aux_z, 2)+pow(aux_x, 2)) #son longitudes auxiliares para el cálculo
    epsilon = acos(aux_z/norm_4_P) #ángulo auxiliar para determinar theta_2
    alpha = acos((pow(L_1, 2)+pow(norm_4_P, 2)-pow(L_2, 2))/(2*L_1*norm_4_P)) #ángulo auxiliar para determinar theta_2
    theta_2 = epsilon - alpha #este es el ángulo de giro del 'hombro' del DOFBot
    theta_3 = 3.1416 - asin((sin(alpha)*sqrt(pow(aux_x, 2) + pow(aux_z, 2)))/(L_2)) #este es el ángulo de giro del 'antebrazo' del DOFBot
    theta_4 = theta_1_P - theta_2 - theta_3 #este es el ángulo de giro de la 'muñeca'
    theta_5 = theta_g #este es el ángulo de giro del gripper, no la apertura del gripper como pinza puesto que se analizó por aparte
    return [ float(theta_1), float(-theta_2), float(-theta_3), float(-theta_4), float(theta_5)]

def gripper_state(theta): #esta función determina los ángulos de apertura del gripper como pinza
    
    return [float(-theta), float(theta), float(-theta), float(theta), float(-theta), float(theta)]



def main(args=None): #Función principal que inicializa ROS 2 y ejecuta el nodo
    rclpy.init(args=args)
    node = DofbotControlNode()
    rclpy.spin(node)
    rclpy.shutdown()

if __name__ == "__main__": #Llamada directa para ejecutar el nodo
    main()

### **Documentación del código de trayectoria. scara_tray_line.py**

A continuación se muestra el código que genera los cálculos de la trayectoria del efector final de un robot SCARA:

![Robot SCARA con 3 juntas rotacionales, robot tipo RRR](Imagenes/SACARAImagen.JPG)

In [ ]:
#!/usr/bin/env python3
#El Shebang indica al sistema que debe iniciar el programa con Python 3

import rclpy #librería principal de ROS 2
from rclpy.node import Node #Se importa la clase Node para crear nodos
from trajectory_msgs.msg import JointTrajectory, JointTrajectoryPoint #permiten enviar trayectorias a controladores de movimiento
from builtin_interfaces.msg import Duration #para especificar tiempos dentro de un mensaje

import time
from math import cos, sin, acos, asin, atan2, sqrt #para las ecuaciones trigonométricas de la cinemática inversa

In [ ]:
class ScaraTrayLineNode(Node): 
    def __init__(self): 
        super().__init__("scara_tray_line_node") #nuevo nodo
        topic_name = "/scara_trajectory_controller/joint_trajectory" #el tópico donde se publica la trayectoria del robot
        self.joints_ = ['link_1_joint', 'link_2_joint', 'link_3_joint'] #se definen las 3 juntas del robot SCARA
        self.lamda_ = 0 #
        self.Tiempo_ejec_ = 10 #se define el tiempo de duración de la trayectoria

        self.scara_tray_pub_ = self.create_publisher(JointTrajectory, topic_name, 10) #se define un nodo publicador que envía mensajes tipo JointTrajectory
        self.tray_timer_ = self.create_timer(1, self.trayectory_cbck) #la variable self.tray_timer_ hace que la función trayectory_cvck se ejecute cada segundo
        self.get_logger().info('Scara activo, trayectoria linea recta') #arroja un mensaje en la terminal indicando que el nodo está activo

    def trayectory_cbck(self): #se define la función trayectory_cbck
        trayectory_msg = JointTrajectory() #trayectory_msg ahora es una variable tipo JointTrajectory()
        trayectory_msg.joint_names = self.joints_ #el objeto trayectory_msg invoca al método joint_names que muestra los nombres de las 3 juntas
        point = JointTrajectoryPoint() #crea un punto de trayectoria


In [ ]:
        #Mientras la trayectoria no termine
        if self.lamda_ <= self.Tiempo_ejec_:
            x_1 = 0.1
            y_1 = 0.6
            theta_1 = 0 #se define la posición del punto inicial P1
            x_2 = 0.3
            y_2 = -0.6
            theta_2 = 1.57 #se define la posición del punto final P2
            solucion = invk_sol(self.lamda_, x_1, y_1,theta_1, x_2, y_2,theta_2) #se invoca la función invk_sol que resuelve la trayectoria
            point.positions = solucion #se asignan los ángulos de la solución al punto de la trayectoria
            point.time_from_start = Duration(sec=1) #define el tiempo desde el inicio para este punto
            trayectory_msg.points.append(point) #agrega el punto o posición al mensaje de la trayectoria
            self.scara_tray_pub_.publish(trayectory_msg) #el nodo scara_tray_pub_ publica la posición
            self.get_logger().info("Postura actual {}".format(solucion)) #muestra un mensaje en la terminal que indica las posiciones actuales de los ángulos
            time.sleep(2) #pausa 2 segundos para simular el paso del tiempo
            self.lamda_ += 1 #incrementa el valor de lamda que determina el progreso de la trayectoria
        elif self.lamda_ > 10: #si lambda alcanza un valor mayor al tiempo de ejecución...
            link_1_joint = 0
            link_2_joint = 0
            link_3_joint = 0 #los ángulos de las juntas los devuelve a cero
            return [float(link_1_joint), float(link_2_joint), float(link_3_joint)] #esta línea indica que la función trayectory_cbck devuelve los valores de los ángulos de las juntas


In [ ]:
def invk_sol(param,x_in, y_in, theta_in, x_fin, y_fin, theta_fin): #la función invk_sol tiene como parámetros los valores de la posición inicial y final, es la función que resuelve la cinemática
    Tiempo_ejec_ = 10 #se ajusta el tiempo de ejecución a 10 segundos
    L_1 = 0.5
    L_2 = 0.5
    L_3 = 0.3 #se definen las longitudes de los eslabones del robot SCARA
    x_P = x_in + (param/Tiempo_ejec_)*(x_fin - x_in)
    y_P = y_in + (param/Tiempo_ejec_)*(y_fin - y_in)
    theta_P = theta_in + (param/Tiempo_ejec_)*(theta_fin - theta_in) #para estos 3 valores del punto P se calcula un incremento lineal en el rango inicial y final que depende del aumento de lamda
    x_3 = x_P - L_3*cos(theta_P) #calcula la abscisa de la junta 3 en el sistema inercial
    y_3 = y_P - L_3*sin(theta_P) #calcula la ordenada de la junta 3 en el sistema inercial
    theta_2 = acos((pow(x_3, 2)+pow(y_3,2)-pow(L_1, 2)-pow(L_2, 2))/(2*L_1*L_2)) #se utiliza ley de cosenos para determinar el ángulo del eslabón 2, en clase se proponía ley de senos
    beta = atan2(y_3, x_3) #en clase este ángulo fue llamado Epsilon y se encuentra entre P_0_1_3 y x_0; donde P_0_1_3 es el vector de posición del efector final que apunta del origen a la junta 3
    psi = acos((pow(x_3, 2)+pow(y_3,2)+pow(L_1, 2)-pow(L_2, 2))/(2*L_1*sqrt(pow(x_3, 2)+pow(y_3,2)))) #en clase este ángulo fue llamado alfa y se encuentra entre el eslabón l1 y P_0_1_3
    theta_1 = beta - psi #gracias a los ángulos calculados anteriormente se puede obtener el ángulo del eslabón 1
    theta_3 = theta_P -theta_1 -theta_2 #se calcula el ángulo del efector final (del eslabón 3)
    return [float(theta_1), float(theta_2), float(theta_3)] #devuelve los valores de los 3 ángulos calculados

def main(args=None): #Inicializa el entorno de ROS 2 y ejecuta el nodo ScaraTrayLineNode()
    rclpy.init(args=args)
    node = ScaraTrayLineNode()
    rclpy.spin(node) #mantiene el nodo activo
    rclpy.shutdown() #permite cerrar ROS 2 y terminar

if __name__ == "__main__": #Llamada directa para ejecutar el nodo
    main()

### **Conclusiones**

- El análisis de ambos robots es distinto primeramente por las características físicas que tiene cada uno. Un robot SCARA maneja de modo distinto su efector final, o al menos es la interpretación que se le puede dar por el código escrito, no tiene un gripper. Pero también sucede que el DOFBot tiene dos grados más de libertad que el robot SCARA pues tiene 6 actuadores, en cambio el SCARA tiene 4 considerando que no hay gripper en él.
- El robot DOFBot tiene un grado de libertad especial que el SCARA no tiene, un giro extra en la muñeca.
- A pesar de lo anterior, el análisis es básicamente el mismo solo que el método fue distinto. Para el robot SCARA se definieron movimientos con base en una sola trayectoria trazada con un punto de inicio y uno de final. En cambio, el DOFBot tuvo diferentes movimientos discretos y secuenciales pero no con una trayectoria en específico, sino que adoptó diferentes posturas solamente.